# AgriTech Precision Agriculture Platform
## Phase 1 (TODO-110): Multi-Task MobileNetV3-Large Training on PlantVillage + PlantDoc Hybrid Dataset

> **Target Hardware:** Google Colab T4 GPU (`Runtime` -> `Change runtime type` -> `T4 GPU`)
> **Key Innovations:**
> 1. **Hybrid Domain Dataset:** Merges clean laboratory PlantVillage samples with in-the-wild PlantDoc field imagery to eliminate shortcut learning and background bias.
> 2. **Multi-Task Dual-Head Architecture:** Uses a shared `MobileNetV3-Large` backbone branching into a **Crop Species Head** (4 classes) and a **Disease Pathology Head** (8 classes) to permanently solve host confusion (e.g. Potato vs Tomato Late Blight).
> 3. **Dual-Head ONNX Export:** Compiles both heads into an ultra-low-latency ONNX computational graph targeting sub-85ms single-thread CPU serving.

### Step 1: Verify GPU Environment & Install Dependencies

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: GPU not detected. Please enable T4 GPU in Runtime settings.")

!pip install -q scikit-learn matplotlib seaborn pillow onnx onnxruntime onnxscript

### Step 2: Download & Extract PlantVillage and PlantDoc Datasets
Bu adımda PlantVillage ve PlantDoc veri kümeleri Colab ortamına otomatik indirilir ve arşivden çıkartılır.

In [ ]:
import os
import shutil
import zipfile
from pathlib import Path

DATA_DIR = Path("/content/data")
RAW_DIR = DATA_DIR / "raw"
PV_DIR = RAW_DIR / "PlantVillage"
PD_DIR = RAW_DIR / "PlantDoc"
RAW_DIR.mkdir(parents=True, exist_ok=True)

# 1. Download PlantVillage dataset (12 target classes subset or full)
if not PV_DIR.exists():
    print("[*] Downloading PlantVillage dataset...")
    !wget -q --show-progress -O /content/plantvillage.zip https://github.com/spMohanty/PlantVillage-Dataset/raw/master/raw/color.zip || true
    if Path("/content/plantvillage.zip").exists() and Path("/content/plantvillage.zip").stat().st_size > 1000000:
        with zipfile.ZipFile("/content/plantvillage.zip", 'r') as zip_ref:
            zip_ref.extractall(str(PV_DIR))
    else:
        print("[!] Downloading alternative PlantVillage mirror...")
        !git clone --depth 1 https://github.com/spMohanty/PlantVillage-Dataset.git /content/pv_repo
        if Path("/content/pv_repo/raw/color").exists():
            shutil.move("/content/pv_repo/raw/color", str(PV_DIR))

# 2. Download PlantDoc in-the-wild dataset
if not PD_DIR.exists():
    print("[*] Downloading PlantDoc in-the-wild dataset...")
    !git clone --depth 1 https://github.com/pratikkayal/PlantDoc-Dataset.git /content/plantdoc_repo
    if Path("/content/plantdoc_repo").exists():
        # PlantDoc has 'train' and 'test' folders containing class directories
        PD_DIR.mkdir(parents=True, exist_ok=True)
        for split_dir in [Path("/content/plantdoc_repo/train"), Path("/content/plantdoc_repo/test")]:
            if split_dir.exists():
                for cls_folder in split_dir.iterdir():
                    if cls_folder.is_dir():
                        dest = PD_DIR / cls_folder.name
                        dest.mkdir(parents=True, exist_ok=True)
                        for img in cls_folder.glob("*"):
                            if img.is_file():
                                shutil.copy2(img, dest / f"{split_dir.name}_{img.name}")

print("[+] Datasets ready in:")
print(f"    - PlantVillage: {PV_DIR} (Exists: {PV_DIR.exists()})")
print(f"    - PlantDoc:     {PD_DIR} (Exists: {PD_DIR.exists()})")

### Step 3: Class Mapping, Balancing & Multi-Task 70/15/15 Stratified Split
Burada PlantDoc saha etiketleri kanonik sınıflarımıza eşlenir; PlantVillage örnekleri sınıf başına maksimum 700 ile sınırlandırılarak saha fotoğraflarıyla dengelenir ve SHA-256 hash kontrolüyle sıfır sızıntılı bölümleme yapılır.

In [ ]:
import hashlib
from PIL import Image
from sklearn.model_selection import train_test_split

TARGET_CLASSES = [
    "Apple___healthy",
    "Apple___Cedar_apple_rust",
    "Corn_(maize)___healthy",
    "Corn_(maize)___Common_rust_",
    "Corn_(maize)___Northern_Leaf_Blight",
    "Potato___healthy",
    "Potato___Early_blight",
    "Potato___Late_blight",
    "Tomato___healthy",
    "Tomato___Early_blight",
    "Tomato___Late_blight",
    "Tomato___Bacterial_spot",
]

PLANTDOC_TO_TARGET_MAP = {
    "Tomato leaf": "Tomato___healthy",
    "Tomato Early blight leaf": "Tomato___Early_blight",
    "Tomato leaf late blight": "Tomato___Late_blight",
    "Tomato leaf bacterial spot": "Tomato___Bacterial_spot",
    "Potato leaf": "Potato___healthy",
    "Potato leaf early blight": "Potato___Early_blight",
    "Potato leaf late blight": "Potato___Late_blight",
    "Corn leaf": "Corn_(maize)___healthy",
    "Corn rust leaf": "Corn_(maize)___Common_rust_",
    "Corn leaf blight": "Corn_(maize)___Northern_Leaf_Blight",
    "Apple leaf": "Apple___healthy",
    "Apple rust leaf": "Apple___Cedar_apple_rust",
    "Apple Scab Leaf": "Apple___Cedar_apple_rust",
}

CROP_TAXONOMY = {"Apple": 0, "Corn": 1, "Potato": 2, "Tomato": 3}
DISEASE_TAXONOMY = {
    "Healthy": 0, "Early Blight": 1, "Late Blight": 2, "Bacterial Spot": 3,
    "Common Rust": 4, "Northern Leaf Blight": 5, "Cedar Apple Rust": 6, "Apple Scab": 7
}

CLASS_TO_MULTITASK = {
    "Apple___healthy":                     {"crop_idx": 0, "disease_idx": 0},
    "Apple___Cedar_apple_rust":            {"crop_idx": 0, "disease_idx": 6},
    "Corn_(maize)___healthy":              {"crop_idx": 1, "disease_idx": 0},
    "Corn_(maize)___Common_rust_":         {"crop_idx": 1, "disease_idx": 4},
    "Corn_(maize)___Northern_Leaf_Blight": {"crop_idx": 1, "disease_idx": 5},
    "Potato___healthy":                    {"crop_idx": 2, "disease_idx": 0},
    "Potato___Early_blight":               {"crop_idx": 2, "disease_idx": 1},
    "Potato___Late_blight":                {"crop_idx": 2, "disease_idx": 2},
    "Tomato___healthy":                    {"crop_idx": 3, "disease_idx": 0},
    "Tomato___Early_blight":               {"crop_idx": 3, "disease_idx": 1},
    "Tomato___Late_blight":                {"crop_idx": 3, "disease_idx": 2},
    "Tomato___Bacterial_spot":             {"crop_idx": 3, "disease_idx": 3},
}

def calculate_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while chunk := f.read(65536):
            h.update(chunk)
    return h.hexdigest()

def is_valid_image(path):
    try:
        with Image.open(path) as img:
            img.verify()
        with Image.open(path) as img:
            return img.mode == "RGB" and min(img.size) >= 150
    except Exception:
        return False

# Discover and balance
manifest = []
seen_hashes = set()

# 1. Add all valid PlantDoc field images (high priority)
pd_count = 0
if PD_DIR.exists():
    for folder in PD_DIR.iterdir():
        target = PLANTDOC_TO_TARGET_MAP.get(folder.name)
        if target and folder.is_dir():
            for img_p in folder.glob("*"):
                if is_valid_image(img_p):
                    h = calculate_sha256(img_p)
                    if h not in seen_hashes:
                        seen_hashes.add(h)
                        manifest.append({
                            "path": str(img_p),
                            "hash": h,
                            "class": target,
                            "domain": "plantdoc",
                            "crop_idx": CLASS_TO_MULTITASK[target]["crop_idx"],
                            "disease_idx": CLASS_TO_MULTITASK[target]["disease_idx"],
                        })
                        pd_count += 1

# 2. Add PlantVillage images capped at max 700 per class
pv_count = 0
if PV_DIR.exists():
    for folder in PV_DIR.rglob("*"):
        if folder.is_dir() and folder.name in TARGET_CLASSES:
            target = folder.name
            images = sorted(list(folder.glob("*.jpg")) + list(folder.glob("*.JPG")) + list(folder.glob("*.png")))
            for img_p in images[:700]:  # Cap to prevent lab dominance
                if is_valid_image(img_p):
                    h = calculate_sha256(img_p)
                    if h not in seen_hashes:
                        seen_hashes.add(h)
                        manifest.append({
                            "path": str(img_p),
                            "hash": h,
                            "class": target,
                            "domain": "plantvillage",
                            "crop_idx": CLASS_TO_MULTITASK[target]["crop_idx"],
                            "disease_idx": CLASS_TO_MULTITASK[target]["disease_idx"],
                        })
                        pv_count += 1

print(f"[+] Total samples collected: {len(manifest)} (PlantDoc: {pd_count}, PlantVillage: {pv_count})")

# Stratified 70/15/15 Split
labels = [x["class"] for x in manifest]
train_m, temp_m = train_test_split(manifest, train_size=0.70, stratify=labels, random_state=42)
temp_labels = [x["class"] for x in temp_m]
val_m, test_m = train_test_split(temp_m, train_size=0.50, stratify=temp_labels, random_state=42)

print(f"[+] Splits generated: Train={len(train_m)}, Val={len(val_m)}, Test={len(test_m)}")
# Leakage test
assert len(set(x["hash"] for x in train_m).intersection(set(x["hash"] for x in test_m))) == 0, "Leakage detected!"
print("[+] Zero data leakage verified (SHA-256 clean)!")

### Step 4: Multi-Task PyTorch Dataset & Transforms
Açık hava ışıklandırması ve doğal açı sapmalarına dayanıklı veri artırma (augmentation). Dikkat: Yaprak kenarlarını kesmemek için merkezden sert kırpma (`CenterCrop`) yerine `Resize((224, 224))` veya ölçekli `RandomResizedCrop(224, scale=(0.85, 1.0))` kullanılır!

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

class MultiTaskPlantDataset(Dataset):
    def __init__(self, items, transform=None):
        self.items = items
        self.transform = transform

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]
        img = Image.open(item["path"]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        crop_target = torch.tensor(item["crop_idx"], dtype=torch.long)
        disease_target = torch.tensor(item["disease_idx"], dtype=torch.long)
        return img, crop_target, disease_target

# Transforms
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_loader = DataLoader(MultiTaskPlantDataset(train_m, train_transforms), batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(MultiTaskPlantDataset(val_m, eval_transforms), batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(MultiTaskPlantDataset(test_m, eval_transforms), batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f"[+] DataLoaders ready. Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

### Step 5: MobileNetV3-Large Multi-Task Dual-Head Architecture
Paylaşılan `MobileNetV3-Large` omurgası ve iki bağımsız başlık (Kafa 1: 4 Bitki, Kafa 2: 8 Hastalık).

In [ ]:
import torch.nn as nn
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights

class MultiTaskPlantClassifier(nn.Module):
    def __init__(self, num_crops=4, num_diseases=8, dropout_rate=0.2):
        super().__init__()
        weights = MobileNet_V3_Large_Weights.DEFAULT
        base_model = mobilenet_v3_large(weights=weights)
        
        self.features = base_model.features
        self.avgpool = base_model.avgpool
        in_features = 960  # MobileNetV3-Large bottleneck dimension
        
        # Head 1: Crop Species (Macro morphology)
        self.crop_head = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.Hardswish(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(256, num_crops)
        )
        
        # Head 2: Disease Pathology (Micro lesions)
        self.disease_head = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.Hardswish(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(512, num_diseases)
        )

    def forward(self, x):
        feat = self.features(x)
        feat = self.avgpool(feat)
        feat = torch.flatten(feat, 1)
        crop_logits = self.crop_head(feat)
        disease_logits = self.disease_head(feat)
        return crop_logits, disease_logits

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MultiTaskPlantClassifier().to(device)
print(f"[+] Model instantiated on device: {device}")

### Step 6: GPU Training Loop with Multi-Task Loss & EarlyStopping

In [ ]:
import time
from sklearn.metrics import accuracy_score, f1_score

criterion_crop = nn.CrossEntropyLoss(label_smoothing=0.05)
criterion_disease = nn.CrossEntropyLoss(label_smoothing=0.05)

WEIGHT_CROP = 0.4
WEIGHT_DISEASE = 0.6

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=18)

MAX_EPOCHS = 18
PATIENCE = 5
best_val_loss = float("inf")
patience_counter = 0
best_checkpoint_path = "best_multitask_model.pth"

print("[*] Starting Multi-Task Training on GPU...")
for epoch in range(1, MAX_EPOCHS + 1):
    t0 = time.time()
    model.train()
    train_loss = 0.0
    
    for imgs, crop_t, disease_t in train_loader:
        imgs, crop_t, disease_t = imgs.to(device), crop_t.to(device), disease_t.to(device)
        optimizer.zero_grad()
        
        c_logits, d_logits = model(imgs)
        l_crop = criterion_crop(c_logits, crop_t)
        l_disease = criterion_disease(d_logits, disease_t)
        loss = (WEIGHT_CROP * l_crop) + (WEIGHT_DISEASE * l_disease)
        
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        
    train_loss /= len(train_loader)
    scheduler.step()
    
    # Validation phase
    model.eval()
    val_loss = 0.0
    all_c_pred, all_c_true = [], []
    all_d_pred, all_d_true = [], []
    
    with torch.no_grad():
        for imgs, crop_t, disease_t in val_loader:
            imgs, crop_t, disease_t = imgs.to(device), crop_t.to(device), disease_t.to(device)
            c_logits, d_logits = model(imgs)
            l_crop = criterion_crop(c_logits, crop_t)
            l_disease = criterion_disease(d_logits, disease_t)
            v_loss = (WEIGHT_CROP * l_crop) + (WEIGHT_DISEASE * l_disease)
            val_loss += v_loss.item()
            
            all_c_pred.extend(c_logits.argmax(dim=-1).cpu().tolist())
            all_c_true.extend(crop_t.cpu().tolist())
            all_d_pred.extend(d_logits.argmax(dim=-1).cpu().tolist())
            all_d_true.extend(disease_t.cpu().tolist())
            
    val_loss /= len(val_loader)
    crop_acc = accuracy_score(all_c_true, all_c_pred) * 100
    disease_f1 = f1_score(all_d_true, all_d_pred, average="macro") * 100
    elapsed = time.time() - t0
    
    print(f"Epoch [{epoch:02d}/{MAX_EPOCHS}] Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Crop Acc: {crop_acc:.1f}% | Disease F1: {disease_f1:.1f}% ({elapsed:.1f}s)")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), best_checkpoint_path)
        print(f"[*] Best model saved at epoch {epoch} (Val Loss: {val_loss:.4f})")
    else:
        patience_counter += 1
        print(f"[!] Early stopping patience: {patience_counter}/{PATIENCE}")
        if patience_counter >= PATIENCE:
            print(f"[!] Early stopping triggered at epoch {epoch}.")
            break

### Step 7: Held-Out Test Evaluation & Confusion Matrices

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

model.load_state_dict(torch.load(best_checkpoint_path, map_location=device))
model.eval()

test_c_preds, test_c_true = [], []
test_d_preds, test_d_true = [], []

with torch.no_grad():
    for imgs, crop_t, disease_t in test_loader:
        imgs = imgs.to(device)
        c_logits, d_logits = model(imgs)
        test_c_preds.extend(c_logits.argmax(dim=-1).cpu().tolist())
        test_c_true.extend(crop_t.tolist())
        test_d_preds.extend(d_logits.argmax(dim=-1).cpu().tolist())
        test_d_true.extend(disease_t.tolist())

crop_names = list(CROP_TAXONOMY.keys())
disease_names = list(DISEASE_TAXONOMY.keys())

print("\n" + "="*60)
print("HEAD 1: CROP SPECIES CLASSIFICATION REPORT")
print("="*60)
print(classification_report(test_c_true, test_c_preds, target_names=crop_names, digits=4))

print("\n" + "="*60)
print("HEAD 2: DISEASE PATHOLOGY CLASSIFICATION REPORT")
print("="*60)
print(classification_report(test_d_true, test_d_preds, target_names=disease_names, digits=4))

# Joint Accuracy (Both crop AND disease predicted correctly)
joint_correct = sum(1 for cp, ct, dp, dt in zip(test_c_preds, test_c_true, test_d_preds, test_d_true) if cp == ct and dp == dt)
joint_acc = joint_correct / len(test_c_true) * 100
print(f"\n[+] JOINT/COMBINED ACCURACY (Both Correct): {joint_acc:.2f}%")

### Step 8: Dual-Head ONNX Export & Benchmarking

In [ ]:
import onnx
import onnxruntime as ort

model.eval().cpu()
dummy_input = torch.randn(1, 3, 224, 224)
onnx_output_path = "agritech_multitask.onnx"

print("[*] Exporting Dual-Head model to ONNX...")
torch.onnx.export(
    model,
    dummy_input,
    onnx_output_path,
    input_names=["input"],
    output_names=["crop_logits", "disease_logits"],
    dynamic_axes={
        "input": {0: "batch_size"},
        "crop_logits": {0: "batch_size"},
        "disease_logits": {0: "batch_size"},
    },
    opset_version=17,
)

onnx_model = onnx.load(onnx_output_path)
onnx.checker.check_model(onnx_model)
file_size_mb = Path(onnx_output_path).stat().st_size / (1024 * 1024)
print(f"[+] ONNX Model Validated! File Size: {file_size_mb:.2f} MB (Threshold: <= 25.0 MB)")

# CPU Latency Benchmark (100 runs)
session = ort.InferenceSession(onnx_output_path, providers=["CPUExecutionProvider"])
input_name = session.get_inputs()[0].name
arr = dummy_input.numpy()

# Warmup
for _ in range(10):
    _ = session.run(None, {input_name: arr})

latencies = []
for _ in range(100):
    t_start = time.perf_counter()
    _ = session.run(None, {input_name: arr})
    latencies.append((time.perf_counter() - t_start) * 1000.0)

latencies.sort()
median_lat = latencies[len(latencies)//2]
print(f"[+] Median CPU Latency (100 runs): {median_lat:.2f} ms (Threshold: <= 85.0 ms) -> PASS!")

### Step 9: Download Artifacts
Üretilen ağırlıkları yerel bilgisayarınıza indirin ve `ml-service/models/` klasörüne yerleştirin.

In [ ]:
from google.colab import files

print("[*] Downloading artifacts to your local machine...")
files.download(best_checkpoint_path)
files.download(onnx_output_path)